In [1]:
# %%
# ═══════════════════════════════════════════════════════════
#  SmolVLM2-2.2B  ×  Kvasir-VQA  —  BASELINE (No Fine-Tuning)
# ═══════════════════════════════════════════════════════════

import os
os.environ["HF_HOME"] = "/mnt/d/huggingface_cache"

import torch
import json
import numpy as np
import re
from PIL import Image
from collections import defaultdict
from datasets import load_dataset
from transformers import AutoProcessor, AutoModelForImageTextToText, BitsAndBytesConfig
from rouge_score import rouge_scorer
from tqdm import tqdm
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.translate.meteor_score import meteor_score
import nltk
nltk.download('wordnet')


[nltk_data] Downloading package wordnet to /home/akanksha/nltk_data...


True

In [6]:


# ── CONFIG — must match your fine-tuning script exactly ───
BASE_MODEL_ID  = "HuggingFaceTB/SmolVLM2-2.2B-Instruct"
OUTPUT_DIR     = "./smolvlm2-baseline-results"
TRAIN_SAMPLES  = 2000
EVAL_SAMPLES   = 50
TEST_SAMPLES   = 5000
MAX_NEW_TOKENS = 64
DEVICE         = "cuda" if torch.cuda.is_available() else "cpu"

os.makedirs(OUTPUT_DIR, exist_ok=True)


# ── 1. LOAD PROCESSOR ─────────────────────────────────────
print("📦 Loading processor...")
processor = AutoProcessor.from_pretrained(BASE_MODEL_ID)


# ── 2. LOAD BASE MODEL IN 4-BIT (no LoRA) ─────────────────
print("🧠 Loading BASE model in 4-bit (no adapter)...")
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_quant_type="nf4",
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

model = AutoModelForImageTextToText.from_pretrained(
    BASE_MODEL_ID,
    quantization_config=bnb_config,
    device_map="auto",
    torch_dtype=torch.bfloat16,
)
model.eval()
print("✅ Base model ready — no LoRA adapter loaded!")


# ── PROMPT TEMPLATE ───────────────────────────────────────
SYSTEM_INSTRUCTION = (
    "Answer in keywords only. Do not use sentences or explanations. "
    "Examples: '0', 'colonoscopy', 'none', 'yes', 'center'."
)

def predict(image: Image.Image, question: str) -> str:
    guided_question = f"{SYSTEM_INSTRUCTION}\n\nQ: {question}"   # ← inject here
    
    messages = [
        {
            "role": "user",
            "content": [
                {"type": "image"},
                {"type": "text", "text": guided_question},
            ],
        }
    ]
    prompt = processor.apply_chat_template(messages, tokenize=False, add_generation_prompt=True)
    inputs = processor(
        text=prompt,
        images=[[image.convert("RGB")]],
        return_tensors="pt",
    ).to(DEVICE)

    with torch.inference_mode():
        output_ids = model.generate(
            **inputs,
            max_new_tokens=MAX_NEW_TOKENS,
            do_sample=False,
        )

    generated = output_ids[0][inputs["input_ids"].shape[-1]:]
    return processor.decode(generated, skip_special_tokens=True).strip()

# ── 4. LOAD SAME TEST SPLIT ───────────────────────────────
# Identical seed + index range as fine-tuned test script
print("\n📂 Loading Kvasir-VQA test samples...")
ds = load_dataset("SimulaMet-HOST/Kvasir-VQA")["raw"]
ds = ds.shuffle(seed=42)
test_start = TRAIN_SAMPLES + EVAL_SAMPLES   # 550
test_end   = test_start + TEST_SAMPLES      # 650
test_ds    = ds.select(range(test_start, test_end))
print(f"✅ Testing on {len(test_ds)} samples (indices {test_start}–{test_end - 1})")


# ── 5. ANSWER NORMALISATION ───────────────────────────────
ANSWER_SYNONYMS = {
    "none":        ["none", "not present", "absent", "nothing", "zero", "there are none", "no instruments"],
    "yes":         ["yes", "correct", "true", "present", "there is", "i can see"],
    "no":          ["no", "false", "not present", "absent", "cannot see"],
    "normal":      ["normal", "no abnormality", "no findings", "unremarkable"],
    "colonoscopy": ["colonoscopy", "colon", "colonoscopic"],
}

def normalize_answer(text: str) -> str:
    text = text.lower().strip()
    for canonical, synonyms in ANSWER_SYNONYMS.items():
        for syn in synonyms:
            if re.search(rf"\b{re.escape(syn)}\b", text):
                return canonical
    return text




📦 Loading processor...
🧠 Loading BASE model in 4-bit (no adapter)...


Loading weights:   0%|          | 0/657 [00:00<?, ?it/s]

✅ Base model ready — no LoRA adapter loaded!

📂 Loading Kvasir-VQA test samples...


Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Resolving data files:   0%|          | 0/31 [00:00<?, ?it/s]

Loading dataset shards:   0%|          | 0/30 [00:00<?, ?it/s]

✅ Testing on 5000 samples (indices 2050–7049)


In [7]:
# ── 6. BATCH EVALUATION ───────────────────────────────────
print("\n🔍 Running baseline inference...")

predictions   = []
ground_truths = []
results       = []

for sample in tqdm(test_ds, desc="Evaluating baseline"):
    image     = sample["image"]
    question  = sample["question"]
    gt_answer = str(sample["answer"]).strip().lower()
    source    = sample["source"]
    img_id    = sample["img_id"]

    pred_answer = predict(image, question).lower()

    gt_norm   = normalize_answer(gt_answer)
    pred_norm = normalize_answer(pred_answer)
    correct   = (gt_norm == pred_norm) or (gt_norm in pred_answer)

    predictions.append(pred_answer)
    ground_truths.append(gt_answer)
    results.append({
        "img_id":    img_id,
        "source":    source,
        "question":  question,
        "gt_answer": gt_answer,
        "gt_norm":   gt_norm,
        "predicted": pred_answer,
        "pred_norm": pred_norm,
        "correct":   correct,
    })


🔍 Running baseline inference...


Evaluating baseline: 100%|██████████| 5000/5000 [34:04<00:00,  2.44it/s]


In [8]:


# ── ROUGE 1 and 2 ──
scorer = rouge_scorer.RougeScorer(["rouge1", "rouge2", "rougeL"], use_stemmer=True)
rouge1_scores = []
rouge2_scores = []
rougeL_scores = []

for gt, pred in zip(ground_truths, predictions):
    scores = scorer.score(gt, pred)
    rouge1_scores.append(scores["rouge1"].fmeasure)
    rouge2_scores.append(scores["rouge2"].fmeasure)
    rougeL_scores.append(scores["rougeL"].fmeasure)

# ── BLEU ──
smoothie = SmoothingFunction().method1
bleu_scores = [
    sentence_bleu(
        [gt.split()],          # reference (list of list of words)
        pred.split(),          # hypothesis (list of words)
        smoothing_function=smoothie
    )
    for gt, pred in zip(ground_truths, predictions)
]

# ── METEOR ──
meteor_scores = [
    meteor_score([gt.split()], pred.split())
    for gt, pred in zip(ground_truths, predictions)
]

# ── PRINT ALL ──
print(f"  ROUGE-1   : {np.mean(rouge1_scores):.4f}")
print(f"  ROUGE-2   : {np.mean(rouge2_scores):.4f}")
print(f"  ROUGE-L   : {np.mean(rougeL_scores):.4f}")
print(f"  BLEU      : {np.mean(bleu_scores):.4f}")
print(f"  METEOR    : {np.mean(meteor_scores):.4f}")

  ROUGE-1   : 0.3528
  ROUGE-2   : 0.0004
  ROUGE-L   : 0.3528
  BLEU      : 0.0594
  METEOR    : 0.1680


In [10]:


# Fixed (proper EM with normalization)
strict_matches = [
    normalize_answer(r["gt_answer"]) == normalize_answer(r["predicted"]) 
    for r in results
]
strict_accuracy = np.mean(strict_matches)
print(f"  Strict Exact Match   : {strict_accuracy:.2%}  ({sum(strict_matches)}/{len(strict_matches)})")

  Strict Exact Match   : 33.06%  (1653/5000)


In [ ]:



# # ── 7. METRICS ────────────────────────────────────────────
# print("\n📊 Computing metrics...")

# norm_correct    = [r["correct"] for r in results]
# norm_accuracy   = np.mean(norm_correct)

# strict_matches  = [r["gt_answer"] == r["predicted"] for r in results]
# strict_accuracy = np.mean(strict_matches)

# scorer = rouge_scorer.RougeScorer(["rougeL"], use_stemmer=True)
# rouge_scores = [
#     scorer.score(gt, pred)["rougeL"].fmeasure
#     for gt, pred in zip(ground_truths, predictions)
# ]
# avg_rouge_l = np.mean(rouge_scores)

# source_stats = defaultdict(lambda: {"correct": 0, "total": 0})
# for r in results:
#     source_stats[r["source"]]["total"]   += 1
#     source_stats[r["source"]]["correct"] += int(r["correct"])

# print("\n" + "=" * 55)
# print("  BASELINE RESULTS  (no fine-tuning)")
# print("=" * 55)
# print(f"  Normalised Accuracy  : {norm_accuracy:.2%}  ({sum(norm_correct)}/{len(norm_correct)})")
# print(f"  Strict Exact Match   : {strict_accuracy:.2%}  ({sum(strict_matches)}/{len(strict_matches)})")
# print(f"  Average ROUGE-L      : {avg_rouge_l:.4f}")
# print("-" * 55)
# print("  Per-Source Breakdown:")
# for src, stat in sorted(source_stats.items()):
#     acc = stat["correct"] / stat["total"]
#     print(f"    {src:<30} {acc:.2%}  ({stat['correct']}/{stat['total']})")
# print("=" * 55)


# # ── 8. QUALITATIVE SAMPLES ────────────────────────────────
# print("\n🔎 Sample Predictions (first 10):\n")
# header = f"{'img_id':<12} {'Question':<40} {'GT':>12} {'GT-norm':>10} {'Pred':>12} {'Pred-norm':>10} {'OK'}"
# print(header)
# print("-" * len(header))
# for r in results[:10]:
#     icon = "✅" if r["correct"] else "❌"
#     print(
#         f"{str(r['img_id']):<12} "
#         f"{r['question'][:39]:<40} "
#         f"{r['gt_answer'][:11]:>12} "
#         f"{r['gt_norm'][:9]:>10} "
#         f"{r['predicted'][:11]:>12} "
#         f"{r['pred_norm'][:9]:>10} "
#         f"{icon}"
#     )


# # ── 9. SAVE RESULTS ───────────────────────────────────────
# output_path = os.path.join(OUTPUT_DIR, "baseline_eval_results.json")
# summary = {
#     "model":               BASE_MODEL_ID,
#     "adapter":             "none (baseline)",
#     "total_samples":       len(results),
#     "normalised_accuracy": round(float(norm_accuracy), 4),
#     "strict_exact_match":  round(float(strict_accuracy), 4),
#     "avg_rouge_l":         round(float(avg_rouge_l), 4),
#     "per_source": {
#         src: {"accuracy": round(s["correct"] / s["total"], 4), **s}
#         for src, s in source_stats.items()
#     },
#     "per_sample_results":  results,
# }
# with open(output_path, "w") as f:
#     json.dump(summary, f, indent=2)

# print(f"\n💾 Results saved to: {output_path}")


# # ── 10. SIDE-BY-SIDE COMPARISON HELPER ───────────────────
# # After running both scripts, load and compare the two JSON files:

# # import json
# # base   = json.load(open("./smolvlm2-baseline-results/baseline_eval_results.json"))
# # ft     = json.load(open("./smolvlm2-kvasir-finetuned/eval_results.json"))
# #
# # print(f"\n{'Metric':<25} {'Baseline':>10} {'Fine-tuned':>12} {'Delta':>10}")
# # print("-" * 60)
# # for key in ["normalised_accuracy", "strict_exact_match", "avg_rouge_l"]:
# #     b, f = base[key], ft[key]
# #     delta = f - b
# #     arrow = "▲" if delta > 0 else "▼"
# #     print(f"  {key:<23} {b:>10.4f} {f:>12.4f} {arrow}{abs(delta):>8.4f}")
